<a href="https://colab.research.google.com/github/Shaifali07/langgraph_learning/blob/main/iterative_workflow_withllm_tweet_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain-groq

In [ ]:
from langgraph.graph import StateGraph, END, START
from typing import TypedDict, Annotated, Literal
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field

In [ ]:
from google.colab import userdata
import os

groq_api_key = userdata.get('GROQ_API_KEY')
os.environ["GROQ_API_KEY"] = groq_api_key

In [ ]:
llm_generator = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.5)
llm_evaluator=ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.5)
llm_optimizer=ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.5)

In [ ]:
class TweetGenerator(TypedDict):
  tweet:str
  topic:str
  evaluate:dict
  iteration:int
  Max_iteration:int

In [ ]:
class evaluateschema(BaseModel):
  decision:Literal["Accepted","NotAceepted"]=Field('Result of the evaluation- accepted or not accepted')
  reasoning:str=Field('Feedback or reasoning for the decision')

In [ ]:
from langchain_core.messages.tool import ToolOutputMixin
def generate_tweet(state:TweetGenerator):
  topic=state["topic"]
  parser=StrOutputParser()
  prompt=ChatPromptTemplate.from_template("you are a tweeter influencer and you need to generate a funny witty tweet on the topic {topic}")
  chain=prompt|llm_generator|parser
  tweet=chain.invoke({"topic":topic})
  return {"tweet":tweet}

def evaluate_tweet(state:TweetGenerator):
  tweet=state["tweet"]
  topic=state["topic"]
  parser=JsonOutputParser(pydantic_object=evaluateschema)
  format_instructions=parser.get_format_instructions()
  prompt=ChatPromptTemplate.from_template('''You are an expert evaluator of sarcastic social media content. Your task is to determine whether a tweet is high-quality, sarcastic, and worth posting.
TWEET:
"{tweet}" on the topic {topic}
tweet must be unique. Evaluation should be done considering whether it is
Clearly sarcastic (subtle or sharp irony, wit, or dry humor), length of the tweet  must not exceed 140 characters
Easy to understand (not confusing sarcasm)
Engaging and attention-grabbing
Not generic or overused
Concise and impactful
{format_instructions}''')
  chain=prompt|llm_evaluator|parser
  evaluation=chain.invoke({"tweet":tweet,"topic":topic, "format_instructions":format_instructions})
  return {"evaluate":evaluation}

def optimize_tweet(state:TweetGenerator):
  tweet=state["tweet"]
  topic=state["topic"]
  evaluate=state["evaluate"]
  iteration=state["iteration"]+1
  parser=StrOutputParser()
  prompt=ChatPromptTemplate.from_template("Consider the {feedback} of an evaluated tweet and optimize a sarcastic tweet {tweet} on topic {topic}")
  chain=prompt|llm_optimizer|parser
  tweet=chain.invoke({"feedback":evaluate["reasoning"],"topic":topic,"tweet":tweet})
  return {"tweet":tweet,"iteration":iteration}

def accepted(state:TweetGenerator):
  print("Tweet Posted!")
  return state

def check_acceptance(state:TweetGenerator)->Literal['optimize_tweet','accepted']:
  evaluate=state["evaluate"]
  iteration=state["iteration"]
  MaxIteration=state["Max_iteration"]
  if evaluate["decision"]=="Accepted" or iteration > MaxIteration:
    return "accepted"
  else:
    return "optimize_tweet"


In [ ]:
graph=StateGraph(TweetGenerator)

graph.add_node("generate_tweet",generate_tweet)
graph.add_node("evaluate_tweet",evaluate_tweet)
graph.add_node("optimize_tweet",optimize_tweet)
graph.add_node("accepted",accepted)

graph.add_edge(START,"generate_tweet")
graph.add_edge("generate_tweet","evaluate_tweet")
graph.add_conditional_edges("evaluate_tweet",check_acceptance)
graph.add_edge("accepted",END)
graph.add_edge("optimize_tweet","evaluate_tweet")

workflow=graph.compile()

initial_state={"topic":"Indian Politics","iteration":1,"Max_iteration":5}

final_state=workflow.invoke(initial_state)

print(final_state)


Tweet Posted!
{'tweet': 'Here\'s an optimized version of the sarcastic tweet, within the 140 character limit:\n\n"Indian politician\'s speech: 1000 words, 0 substance. \'Jab tak suraj chandrama...\' Yeah, good luck with that, PM #IndianPolitics #Satire"\n\nHowever, if you\'d like to expand on the tweet while keeping it engaging and within the character limit, here\'s another option:\n\n"PM\'s speech: where \'Jab tak suraj chandrama...\' fills the void of actual promises #IndianPolitics #Satire"\n\nOr, if you\'d like to add a bit of humor:\n\n"Indian politician\'s speech: where \'Jab tak suraj chandrama...\' is code for \'I\'ve got nothing to say\' #IndianPolitics #Satire"\n\nThese optimized tweets aim to:\n\n1. Stay within the 140 character limit\n2. Retain the sarcastic tone\n3. Use relevant hashtags (#IndianPolitics #Satire) to reach a wider audience\n4. Add a bit of humor to engage the audience\n5. Keep the core message concise and attention-grabbing', 'topic': 'Indian Politics', 'e

In [ ]:
with open("graph_iterative_workflow_with_llm.png", "wb") as f:
    f.write(workflow.get_graph().draw_mermaid_png())